# NLP Term Project — Kaggle 실행 노트북

**과제 지정 환경: Python 3.10.12 / torch 2.5.1**

## 실행 순서

| 셀 | 내용 | 비고 |
|---|---|---|
| Cell 1 | 환경 확인 | Python/GPU 상태 점검 |
| Cell 2 | torch 2.5.1 설치 | **실행 후 커널 재시작 필수** |
| Cell 3 | 패키지 설치 | 재시작 후 실행 |
| Cell 4 | 프로젝트 경로 설정 | GitHub clone 포함 |
| Cell 5 | 데이터 파일 확인 | |
| Cell 6 | refresh_data | 식단·셔틀·학사일정 갱신 |
| Cell 7 | ChromaDB 구축 | 2~5분 소요 |
| Cell 8 | Task 2 | chat_output.json 생성 |
| Cell 9 | Optional | realtime_output.json 생성 |
| Cell 10 | Task 1 | cls_output.json 생성 |
| Cell 11 | Gradio UI | 공개 URL 출력 |
| Cell 12 | Output 검증 | 최종 확인 |
| Cell 13 | requirements.txt | pip freeze → git push |

> ⚠️ **중요**: Cell 2 실행 후 반드시 `Run > Restart Session` 후 Cell 3부터 실행하세요.

In [ ]:
# ─── Cell 1: 환경 확인 ─────────────────────────────────────────────
# Python 버전, GPU, CUDA 상태를 확인합니다.
# 과제 지정: Python 3.10.12 / torch 2.5.1
import sys, subprocess

print('=' * 60)
print(f'Python : {sys.version}')
print(f'Platform: {sys.platform}')

# GPU 확인 (nvidia-smi)
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f'GPU    : {r.stdout.strip()}')
else:
    print('GPU    : 없음 — Kaggle Accelerator(T4/P100) 설정 확인 필요')

# torch 현재 버전
try:
    import torch
    print(f'torch  : {torch.__version__}')
    print(f'CUDA   : {torch.version.cuda}')
    print(f'GPU OK : {torch.cuda.is_available()}')
except ImportError:
    print('torch  : 미설치 — Cell 2 실행 필요')

# Python 버전 경고
ver = sys.version_info
if (ver.major, ver.minor) == (3, 10):
    print('OK   Python 3.10.x — 과제 지정 범위 내')
else:
    print(f'WARN Python {sys.version.split()[0]} — 과제 지정: 3.10.12')
    print('     코드 동작 가능. requirements.txt에 실제 버전 기록됨.')
print('=' * 60)

In [ ]:
# ─── Cell 2: torch 2.5.1 설치 ──────────────────────────────────────
# 실행 후 반드시 커널을 재시작하고 Cell 3부터 이어서 실행하세요!
import sys, subprocess

try:
    import torch
    current = torch.__version__
except ImportError:
    current = 'not_installed'

print(f'현재 torch: {current}')

if current.startswith('2.5.1'):
    print('OK  torch 2.5.1 이미 설치됨 — 커널 재시작 불필요, Cell 3 진행')
else:
    # CUDA 버전 자동 감지 (Kaggle 기본: CUDA 12.1)
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    # Kaggle 2025~2026 기본 CUDA 12.1 → cu121
    cuda_tag = 'cu121'
    print(f'CUDA wheel: {cuda_tag}')
    print(f'torch==2.5.1+{cuda_tag} 설치 중 (1~3분)...')

    result = subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        f'torch==2.5.1+{cuda_tag}',
        f'torchvision==0.20.1+{cuda_tag}',
        f'torchaudio==2.5.1+{cuda_tag}',
        '--index-url', f'https://download.pytorch.org/whl/{cuda_tag}',
        '--quiet',
    ])

    if result.returncode == 0:
        print()
        print('설치 완료!')
        print('★' * 52)
        print('  커널 재시작이 필요합니다!')
        print('  Kaggle: Run > Restart Session 클릭')
        print('  재시작 후 Cell 3부터 순서대로 실행하세요.')
        print('★' * 52)
    else:
        print('FAIL 설치 실패')
        print('cu124로 재시도: torch==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124')

## ⚠️ 커널 재시작 후 Cell 3부터 실행

Cell 2에서 torch 2.5.1 설치를 완료했다면:

1. Kaggle 상단 메뉴 **Run > Restart Session** 클릭
2. 팝업에서 **Restart** 확인
3. **Cell 3부터** 순서대로 실행

> torch가 이미 2.5.1이면 재시작 없이 바로 Cell 3으로 진행하세요.

In [ ]:
# ─── Cell 3: 패키지 설치 ────────────────────────────────────────────
# 커널 재시작 후 이 셀부터 실행하세요.
# torch는 이미 설치됐으므로 덮어쓰지 않습니다.
import sys, subprocess

packages = [
    # Transformers / 양자화
    'transformers==4.44.2', 'accelerate==0.34.2', 'bitsandbytes==0.43.3',
    'peft==0.12.0', 'datasets==2.21.0', 'trl==0.9.6',
    # RAG / VectorDB
    'sentence-transformers==3.0.1', 'chromadb==0.5.5',
    'langchain==0.2.16', 'langchain-community==0.2.17',
    'rank-bm25==0.2.2', 'kiwipiepy==0.17.1',
    # 크롤링
    'beautifulsoup4==4.12.3', 'requests==2.32.3',
    'lxml==5.3.0', 'pdfplumber==0.11.4',
    # 분류기 / UI / 유틸
    'scikit-learn==1.5.1', 'gradio==4.42.0',
    'tqdm==4.66.5', 'python-dotenv==1.0.1',
]

for pkg in packages:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
                       capture_output=True)
    print(f'{'OK  ' if r.returncode == 0 else 'FAIL'} {pkg}')

# torch 2.5.1 보호 확인 (다른 패키지가 덮어쓰는지 체크)
import torch
tv = torch.__version__
if tv.startswith('2.5.1'):
    print(f'OK   torch {tv} 유지됨 (과제 지정 버전)')
else:
    print(f'WARN torch {tv} — 버전이 변경됨. Cell 2 재실행 후 커널 재시작 필요.')

In [ ]:
# ─── Cell 4: 프로젝트 경로 설정 ─────────────────────────────────────
# 프로젝트 루트를 자동으로 감지하고, 없으면 GitHub에서 clone합니다.
import sys, os, subprocess
from pathlib import Path

# GitHub remote URL 자동 감지 (git config에서)
r = subprocess.run(['git', 'remote', 'get-url', 'origin'],
                   capture_output=True, text=True)
REPO_URL = r.stdout.strip() if r.returncode == 0 else \
           'https://github.com/여기에본인아이디/NLP_Term_Project.git'
print(f'REPO: {REPO_URL}')

# 프로젝트 루트 자동 감지
candidates = [
    Path('/kaggle/working/NLP_Term_Project'),
    Path('/kaggle/working'),
    Path('/content/NLP_Term_Project'),
    Path('/content/drive/MyDrive/NLP_Term_Project'),
    Path.cwd(),
]
ROOT = None
for p in candidates:
    if (p / 'src').exists() and (p / 'data').exists():
        ROOT = p.resolve()
        break

# 없으면 GitHub clone
if ROOT is None:
    target = Path('/kaggle/working/NLP_Term_Project')
    print(f'프로젝트 없음 — clone 중: {REPO_URL}')
    os.system(f'git clone {REPO_URL} {target}')
    ROOT = target.resolve()
else:
    # 최신 코드 pull
    print('git pull 중...')
    os.system(f'git -C {ROOT} pull origin main')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
(ROOT / 'outputs').mkdir(exist_ok=True)

print(f'PROJECT ROOT : {ROOT}')
print(f'outputs/     : {ROOT / "outputs"}')
print(f'cwd          : {os.getcwd()}')

In [ ]:
# ─── Cell 5: 데이터 파일 확인 ───────────────────────────────────────
import json
from pathlib import Path

required = {
    'data/test_cls.json':           'Task1 테스트',
    'data/test_chat.json':          'Task2 테스트',
    'data/test_realtime.json':      'Optional 테스트',
    'data/train.json':              '학습 데이터',
    'data/valid.json':              '검증 데이터',
    'data/processed/chunks.json':   'RAG 청크',
    'src/chatbot_model.py':         'Task2 스크립트',
    'src/realtime_model.py':        'Optional 스크립트',
    'src/chatbot_ui.py':            'Gradio UI',
    'classifier_박연진.ipynb':       'Task1 분류기',
}

missing = []
for path, desc in required.items():
    p = Path(path)
    if p.exists():
        if path.endswith('.json'):
            with open(p, encoding='utf-8') as f:
                d = json.load(f)
            n = len(d) if isinstance(d, list) else 'dict'
            print(f'OK   {path} ({n}건)')
        else:
            print(f'OK   {path}')
    else:
        print(f'MISS {path}  ← {desc}')
        missing.append(path)

print()
if missing:
    print(f'누락 파일 {len(missing)}개 — git pull 또는 데이터 확인 필요')
else:
    print('OK  모든 필수 파일 확인됨')

In [ ]:
# ─── Cell 6: 실시간 데이터 갱신 ─────────────────────────────────────
# 식단, 셔틀버스, 학사일정, 공지사항 최신 데이터를 크롤링합니다.
import os

print('데이터 갱신 중...')
ret = os.system('python scripts/refresh_data.py')

if ret == 0:
    print('OK  refresh_data 완료')
else:
    print(f'WARN refresh_data 일부 실패 (exit {ret}) — 기존 데이터로 진행 가능')

In [ ]:
# ─── Cell 7: ChromaDB 구축 (2~5분 소요) ────────────────────────────
# 이미 chroma_db가 있으면 건너뜁니다.
import os
from pathlib import Path

chroma_path = Path('chroma_db')
files = list(chroma_path.iterdir()) if chroma_path.exists() else []

if files:
    print(f'chroma_db 존재 ({len(files)}개 파일) — 구축 건너뜀')
    print('재구축: import shutil; shutil.rmtree("chroma_db") 후 이 셀 재실행')
else:
    print('ChromaDB 구축 중 (2~5분)...')
    ret1 = os.system('python scripts/inject_faq.py --reset')
    ret2 = os.system('python src/vectordb/build_db.py')
    if ret1 == 0 and ret2 == 0:
        files_after = list(chroma_path.iterdir()) if chroma_path.exists() else []
        print(f'OK  ChromaDB 구축 완료 ({len(files_after)}개 파일)')
    else:
        print('FAIL ChromaDB 구축 실패 — 오류 메시지 확인 필요')

In [ ]:
# ─── Cell 8: Task 2 — chat_output.json 생성 ─────────────────────────
import os, json
from pathlib import Path

print('Task 2 실행 중 (Qwen2.5-3B 로드 ~ 3분)...')
ret = os.system(
    'python src/chatbot_model.py '
    '--input data/test_chat.json '
    '--output outputs/chat_output.json'
)

out = Path('outputs/chat_output.json')
if ret == 0 and out.exists():
    with open(out, encoding='utf-8') as f:
        data = json.load(f)
    mock = sum(1 for d in data if '[MOCK]' in str(d.get('model', '')))
    status = f'WARN MOCK {mock}건 포함' if mock else 'OK  '
    print(f'{status} chat_output.json: {len(data)}건')
    if mock:
        print('     MOCK → ChromaDB 구축 확인 후 재실행 필요')
    print()
    for item in data[:2]:
        print(f'  Q: {item.get("user", item.get("question", ""))[:60]}')
        print(f'  A: {item.get("model", "")[:80]}')
        print()
else:
    print(f'FAIL chat_output.json 생성 실패 (exit {ret})')

In [ ]:
# ─── Cell 9: Optional — realtime_output.json 생성 ───────────────────
import os, json
from pathlib import Path

print('Optional 실행 중...')
ret = os.system(
    'python src/realtime_model.py '
    '--input data/test_realtime.json '
    '--output outputs/realtime_output.json'
)

out = Path('outputs/realtime_output.json')
if ret == 0 and out.exists():
    with open(out, encoding='utf-8') as f:
        data = json.load(f)
    print(f'OK  realtime_output.json: {len(data)}건')
    for item in data:
        print(f'  Q: {item.get("user", "")[:60]}')
        print(f'  A: {item.get("model", "")[:80]}')
        print()
else:
    print(f'FAIL realtime_output.json 생성 실패 (exit {ret})')

In [ ]:
# ─── Cell 10: Task 1 — cls_output.json 생성 ─────────────────────────
# classifier_박연진.ipynb 를 nbconvert로 실행합니다.
import subprocess, sys, os, json
from pathlib import Path

# nbconvert 설치 확인
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'jupyter', 'nbconvert', '--quiet'], capture_output=True)

nb_path = Path('classifier_박연진.ipynb')
if not nb_path.exists():
    print('FAIL classifier_박연진.ipynb 없음 — Cell 4(git pull) 확인')
else:
    print('classifier_박연진.ipynb 실행 중 (KLUE-BERT 학습, 10~30분)...')
    result = subprocess.run([
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to', 'notebook',
        '--execute',
        '--ExecutePreprocessor.timeout=3600',
        '--inplace',
        str(nb_path),
    ], capture_output=True, text=True)

    out = Path('outputs/cls_output.json')
    if result.returncode == 0 and out.exists():
        with open(out, encoding='utf-8') as f:
            data = json.load(f)
        print(f'OK  cls_output.json: {len(data)}건')
        print(f'    sample: {data[0]}')
    elif result.returncode == 0:
        print('WARN nbconvert 성공했지만 cls_output.json 없음')
        print('     classifier_박연진.ipynb 내 outputs 경로 확인 필요')
    else:
        print('FAIL nbconvert 실패')
        print(result.stderr[-800:])

In [ ]:
# ─── Cell 11: Gradio UI 실행 ────────────────────────────────────────
# share=True로 공개 URL을 생성합니다 (https://xxxx.gradio.live).
# Ctrl+C 또는 Kaggle Stop 버튼으로 종료.
import subprocess, sys, time
from pathlib import Path

ui_path = Path('src/chatbot_ui.py')
code = ui_path.read_text(encoding='utf-8')

# share=True 자동 적용
if 'share=True' not in code:
    code = code.replace('demo.launch(', 'demo.launch(share=True, ')
    ui_path.write_text(code, encoding='utf-8')
    print('share=True 적용 완료')

print('Gradio UI 시작 중... (모델 로드 1~3분 소요)')
print('공개 URL(gradio.live)이 출력되면 접속하세요.')
print()

proc = subprocess.Popen(
    [sys.executable, 'src/chatbot_ui.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for _ in range(240):  # 최대 4분 대기
    line = proc.stdout.readline()
    if line:
        print(line.rstrip())
    if 'gradio.live' in line or 'Running on public URL' in line:
        print()
        print('UI 접속 가능! 위 URL을 브라우저에서 여세요.')
        break
    time.sleep(1)

In [ ]:
# ─── Cell 12: Output 파일 검증 ──────────────────────────────────────
import json
from pathlib import Path

outputs = [
    ('outputs/chat_output.json',     'Task 2',   True),
    ('outputs/realtime_output.json', 'Optional', True),
    ('outputs/cls_output.json',      'Task 1',   False),
]

print('=' * 60)
all_ok = True
for path, label, check_mock in outputs:
    p = Path(path)
    if not p.exists():
        print(f'MISS [{label:8}] {path}')
        all_ok = False
        continue
    with open(p, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data)
    if check_mock:
        mock = sum(1 for d in data if '[MOCK]' in str(d.get('model', '')))
        if mock:
            print(f'WARN [{label:8}] {path}: {n}건 (MOCK {mock}건 포함)')
        else:
            print(f'OK   [{label:8}] {path}: {n}건')
    else:
        print(f'OK   [{label:8}] {path}: {n}건')
print('=' * 60)
print('모든 output 정상' if all_ok else '일부 누락 — 해당 셀 재실행 필요')

In [ ]:
# ─── Cell 13: requirements.txt 생성 + git push ──────────────────────
# pip freeze 결과로 requirements.txt를 교체합니다.
import subprocess, sys, os
from datetime import datetime
from pathlib import Path

# pip freeze 실행
r = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                   capture_output=True, text=True)
lines = r.stdout.strip().split('\n')

# 헤더 추가
header = '\n'.join([
    f'# 생성: {datetime.now().isoformat()}',
    f'# 실행 환경: Python {sys.version.split()[0]} / Kaggle Notebook',
    '# 과제 지정: Python 3.10.12 / torch 2.5.1',
    '',
])
Path('requirements.txt').write_text(header + r.stdout, encoding='utf-8')

# torch 버전 확인
torch_line = next((l for l in lines if l.startswith('torch==')), '없음')
ok = 'OK  ' if '2.5.1' in torch_line else 'WARN'
print(f'{ok} {torch_line}')
print(f'requirements.txt 저장 ({len(lines)}개 패키지)')

# git commit + push
print()
print('git push 중...')
os.system('git config user.email "auto@kaggle" 2>/dev/null')
os.system('git config user.name "Kaggle" 2>/dev/null')
os.system('git add outputs/ requirements.txt')
os.system('git commit -m "feat: Kaggle 실행 결과 업데이트"')
os.system('git push origin main')
print()
print('완료! GitHub에서 outputs/ 폴더를 확인하세요.')